# 02 - Doubly robust estimation with AIPW

Notebook 01 adjusted for confounding two ways: by modelling the outcome
(g-computation) and by modelling treatment assignment (IPW). Each is consistent
only if *its* model is right, and you rarely know which one is.

AIPW combines them so that the estimate is consistent if **either** model is
correct. This notebook demonstrates that property by deliberately breaking each
model in turn and watching what survives.

## Causal question

The same question as notebook 01: what is the average effect of the programme on
the outcome, given that uptake depends on covariates?

## Data and design

- **Unit of analysis:** one individual.
- **Treatment:** `treatment`, binary.
- **Outcome:** `outcome`, continuous.
- **Covariates:** `x1`, `x2`, `x3`, measured pre-treatment.

To test double robustness we need a way to corrupt one model without touching
the other. We do that by feeding an estimator a *subset* of the covariates: a
model that cannot see `x2` is misspecified in a controlled, well-understood
way.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC_PATH = PROJECT_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

import numpy as np
import pandas as pd

from causal_inference_lab.data_generators import make_confounded_binary_treatment
from causal_inference_lab.estimators import aipw_ate, difference_in_means, g_computation_ate, ipw_ate
from causal_inference_lab.uncertainty import bootstrap_ate

COVARIATES = ["x1", "x2", "x3"]

dataset = make_confounded_binary_treatment(n=5_000, seed=42)
data = dataset.data

print(f"observations: {len(data):,}")
print(f"true ATE:     {dataset.true_ate:.3f}")
print(f"naive:        {difference_in_means(data).estimate:.3f}")

**Interpretation.** The same confounded setup as notebook 01: the raw contrast
of 3.80 nearly doubles the true effect of 1.99.

## Estimand

The **average treatment effect (ATE)**, as in notebook 01. AIPW targets the same
quantity as IPW and g-computation — it is a more robust route to it, not a
different destination.

## Identification assumptions

1. **Conditional ignorability** given the covariates.
2. **Overlap.**
3. **Consistency and no interference.**
4. **At least one nuisance model is correctly specified** — either the outcome
   model or the propensity model.

Assumption 4 is what AIPW buys, and it is genuinely weaker than what the
single-model estimators require. It is not the same as "no assumptions about
functional form": if both models are wrong, AIPW has no guarantee, which
notebook 07 demonstrates at some cost.

## Estimation

All three adjusted estimators with the full covariate set, so the baseline is
the correctly-specified case.

In [ ]:
correct = pd.DataFrame(
    [
        ("g-computation", g_computation_ate(data, covariates=COVARIATES).estimate),
        ("IPW", ipw_ate(data, covariates=COVARIATES).estimate),
        ("AIPW", aipw_ate(data, covariates=COVARIATES).estimate),
    ],
    columns=["estimator", "estimate"],
)
correct["error"] = correct["estimate"] - dataset.true_ate

print(f"true ATE: {dataset.true_ate:.3f}\n")
print(correct.to_string(index=False, float_format=lambda v: f"{v:.3f}"))

**Interpretation.** With all covariates available every estimator does well,
within 0.11 of the truth. Nothing distinguishes them here, which is exactly why
this case cannot tell you which to prefer. The interesting behaviour only
appears once a model is wrong.

## Diagnostics

The diagnostic that matters for a doubly robust estimator is whether the
robustness claim actually holds. We break one model at a time by restricting the
covariates it can see, and compare against the single-model estimators that rely
on it.

`x2` is a strong confounder — notebook 01 measured its standardized mean
difference at −0.57 — so hiding it is a substantial misspecification.

In [ ]:
partial = ["x1", "x3"]  # x2 hidden: the model cannot represent a real confounder

scenarios = pd.DataFrame(
    [
        (
            "both models correct",
            g_computation_ate(data, covariates=COVARIATES).estimate,
            ipw_ate(data, covariates=COVARIATES).estimate,
            aipw_ate(data, covariates=COVARIATES).estimate,
        ),
        (
            "both models missing x2",
            g_computation_ate(data, covariates=partial).estimate,
            ipw_ate(data, covariates=partial).estimate,
            aipw_ate(data, covariates=partial).estimate,
        ),
    ],
    columns=["scenario", "g-computation", "IPW", "AIPW"],
)
print(f"true ATE: {dataset.true_ate:.3f}\n")
print(scenarios.to_string(index=False, float_format=lambda v: f"{v:.3f}"))

**Interpretation.** Dropping `x2` from *both* nuisance models pushes every
estimator far from the truth, AIPW included. This is the honest boundary of
double robustness: it protects against one model being wrong, not against a
confounder being absent from the analysis altogether.

Note that this is not really a misspecification test — hiding `x2` from both
models violates conditional ignorability itself, because `x2` is a genuine
confounder. No estimator in this repository can survive that, and any that
appeared to would be doing so by luck.

The real test of double robustness needs `x2` present in one model and absent
from the other. `aipw_ate` takes a single covariate list, so we construct the
estimator by hand: fit the outcome model on the full covariates, the propensity
model on the reduced set, and combine them in the AIPW formula.

In [ ]:
from sklearn.linear_model import LinearRegression, LogisticRegression


def aipw_by_hand(frame: pd.DataFrame, outcome_covariates, propensity_covariates) -> float:
    """AIPW with independently chosen covariate sets for each nuisance model."""
    treatment = frame["treatment"].to_numpy()
    outcome = frame["outcome"].to_numpy()

    propensity = (
        LogisticRegression(max_iter=1_000)
        .fit(frame[propensity_covariates], treatment)
        .predict_proba(frame[propensity_covariates])[:, 1]
    )
    propensity = np.clip(propensity, 0.01, 0.99)

    features = frame[outcome_covariates].to_numpy()
    treated_fit = LinearRegression().fit(features[treatment == 1], outcome[treatment == 1])
    control_fit = LinearRegression().fit(features[treatment == 0], outcome[treatment == 0])
    mu1, mu0 = treated_fit.predict(features), control_fit.predict(features)

    augmented = (
        mu1
        - mu0
        + treatment * (outcome - mu1) / propensity
        - (1 - treatment) * (outcome - mu0) / (1 - propensity)
    )
    return float(np.mean(augmented))


rows = [
    ("outcome correct, propensity broken", aipw_by_hand(data, COVARIATES, partial)),
    ("outcome broken, propensity correct", aipw_by_hand(data, partial, COVARIATES)),
    ("both correct", aipw_by_hand(data, COVARIATES, COVARIATES)),
    ("both broken", aipw_by_hand(data, partial, partial)),
]
robustness = pd.DataFrame(rows, columns=["scenario", "AIPW estimate"])
robustness["error"] = robustness["AIPW estimate"] - dataset.true_ate

print(f"true ATE: {dataset.true_ate:.3f}\n")
print(robustness.to_string(index=False, float_format=lambda v: f"{v:.3f}"))

**Interpretation.** This is the property the estimator is named for. With either
nuisance model given the full covariate set, AIPW stays close to the truth even
though the other model is missing a real confounder. Only when both are
restricted does it fail.

The asymmetry between the two middle rows is worth noticing: the two "one model
correct" cases do not perform identically. Double robustness is a consistency
guarantee, not a promise of equal finite-sample accuracy, and which model you
get right still matters for how much error remains.

## Uncertainty

Bootstrapping the correctly-specified AIPW estimator, refitting both nuisance
models on every resample.

In [ ]:
interval = bootstrap_ate(
    data,
    estimator=aipw_ate,
    covariates=COVARIATES,
    n_bootstrap_samples=300,
    seed=42,
)

print(f"AIPW estimate: {interval.estimate:.3f}")
print(f"95% interval:  [{interval.lower:.3f}, {interval.upper:.3f}]")
print(f"true ATE:      {dataset.true_ate:.3f}")
print(f"covers truth:  {interval.lower <= dataset.true_ate <= interval.upper}")

broken = float(robustness.loc[robustness["scenario"] == "both broken", "AIPW estimate"].iloc[0])
print(f"\ninterval width:                    {interval.upper - interval.lower:.3f}")
print(f"error when both models are broken: {abs(broken - dataset.true_ate):.3f}")

**Interpretation.** The interval is narrow — width 0.13 — and it covers the
truth here, unlike the IPW interval in notebook 01.

The comparison underneath is the point. Sampling uncertainty spans about 0.13.
Getting the covariate set wrong costs roughly 1.5. The dominant risk in this
analysis is specification, by an order of magnitude, and no amount of bootstrap
resampling would reveal it.

## Limitations

- **Double robustness is not robustness to omitted confounders.** Both nuisance
  models must be built from covariates sufficient for ignorability. Dropping a
  true confounder from both defeats the estimator entirely, as shown above.
- **The demonstration uses covariate omission as a stand-in for
  misspecification.** Real misspecification is usually about functional form —
  see notebook 07, where both models are wrong in that sense and AIPW lands
  further from the truth than doing nothing.
- **The hand-built estimator is illustrative.** It clips propensities at 0.01
  and does no cross-fitting, so it is not a production estimator.
- **Both nuisance models are linear**, which suits this generator.
- **The interval assumes correct specification.** It is a lower bound on total
  uncertainty.